# Week 7 Facial Emotion Recognition

**Track B:** frozen MobileNetV2 embeddings with Logistic Regression and Linear SVM.

This notebook follows the laboratory milestones: dataset verification, detection, feature extraction, training, evaluation, and two held-out predictions. Metrics appear only after execution. Predicted categories describe dataset expression labels, not proof of internal feelings.

**Source:** [FER2013 folder-format dataset](https://www.kaggle.com/datasets/msambare/fer2013). This is a repackaged train/test protocol, not a claim of original PrivateTest-only evaluation.

**Team members and roles:** fill in before submission.


In [ ]:
from pathlib import Path
import os, sys, json, random, importlib.metadata
ROOT = Path.cwd().resolve()
if not (ROOT / 'src').is_dir():
    ROOT = ROOT.parent
assert (ROOT / 'src').is_dir(), 'Open this notebook from the project root or notebooks directory.'
sys.path.insert(0, str(ROOT))
os.environ.setdefault('KERAS_HOME', str(ROOT / '.cache/keras'))
os.environ.setdefault('MPLCONFIGDIR', str(ROOT / '.cache/matplotlib'))
os.environ.setdefault('TF_CPP_MIN_LOG_LEVEL', '2')
# Restrict CPU thread oversubscription on laptops.
os.environ.setdefault('OMP_NUM_THREADS', '4')
os.environ.setdefault('TF_NUM_INTRAOP_THREADS', '4')
os.environ.setdefault('TF_NUM_INTEROP_THREADS', '2')
DATA_ROOT = Path(os.environ.get('FER_DATA_ROOT', ROOT / 'data')).expanduser()
SEED = 42
BATCH_SIZE = 32
ARTIFACTS = ROOT / 'artifacts'
OUTPUTS = ROOT / 'outputs'
for folder in [ARTIFACTS, OUTPUTS / 'figures', OUTPUTS / 'metrics', OUTPUTS / 'submission']:
    folder.mkdir(parents=True, exist_ok=True)
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf
from IPython.display import display
from src.data import CLASSES, scan_dataset, audit_dataset, validation_partition, prepare_training_data
from src.features import FeatureConfig, FacePreprocessor, FeatureExtractor
from src.training import train_models, evaluate_models
from src.predict import save_bundle, EmotionPredictor, demonstrate
from src.plots import sample_grid, error_grid
random.seed(SEED)
np.random.seed(SEED)
tf.keras.utils.set_random_seed(SEED)
tf.config.experimental.enable_op_determinism()
versions = {p: importlib.metadata.version(p) for p in ['numpy','pandas','scikit-learn','opencv-python','tensorflow','keras','matplotlib','joblib']}
versions['python'] = sys.version
(ARTIFACTS / 'environment.json').write_text(json.dumps(versions, indent=2))
display(pd.Series(versions, name='version'))
print('Dataset:', DATA_ROOT)
print('TensorFlow devices:', tf.config.list_physical_devices())


## Materials and split verification

Keep the supplied test folder separate throughout development. The audit checks readable 48×48 images, seven classes, and decoded-pixel duplicates. Reports are written before any blocking failure. The explicit policy keeps the complete supplied test set, removes training overlap with test, removes conflicting training-label groups, and keeps one copy of each remaining training image. Exclusions are recorded; source files are never modified. Test-only duplicates and label conflicts remain a limitation. An exact-pixel audit does not rule out near-duplicate images or repeated identities.


In [ ]:
manifest = scan_dataset(DATA_ROOT)
counts, audit = audit_dataset(manifest, ARTIFACTS / 'audit', duplicate_policy='report')
display(counts)
display(pd.Series(audit, name='audit'))
train_rows, test_rows, effective_counts = prepare_training_data(manifest, ARTIFACTS / 'audit')
print('Explicit duplicate policy: filter training only; preserve the full supplied test set.')
display(effective_counts)
print((ARTIFACTS / 'audit/duplicate_policy.json').read_text())
fig = sample_grid(train_rows, per_class=3, seed=SEED)
fig.savefig(OUTPUTS / 'figures/class_samples.png', dpi=150, bbox_inches='tight')
plt.show()


## Milestone 1 Face detection and visual check

Run Haar detection on an enlarged image and map the largest detected box back to the source coordinates. Green boxes are real detections. A failed detection on an already cropped FER image uses the full image and is explicitly marked **fallback**. Fallbacks do not count as detection successes. Inspect multiple classes before proceeding.


In [ ]:
config = FeatureConfig()
preprocessor = FacePreprocessor(config)
fig = sample_grid(train_rows, preprocessor=preprocessor, per_class=3, seed=SEED)
fig.savefig(OUTPUTS / 'figures/face_detection.png', dpi=150, bbox_inches='tight')
plt.show()


## Milestone 2 CNN feature extraction

Convert each crop to 224×224, replicate grayscale to three channels, and use MobileNetV2's `preprocess_input`. Frozen ImageNet weights with global average pooling produce 1,280 numbers per image. No CNN fine-tuning is performed. Batches avoid loading all resized images into memory; a content- and configuration-bound cache avoids repeated extraction.

The first run downloads pretrained weights. Feature extraction can take substantial time on a CPU.


In [ ]:
extractor = FeatureExtractor(config)
X_train, train_detection = extractor.extract(train_rows, ARTIFACTS / 'features', BATCH_SIZE)
X_test, test_detection = extractor.extract(test_rows, ARTIFACTS / 'features', BATCH_SIZE)
print('Train features:', X_train.shape, '| Test features:', X_test.shape)
detections = pd.concat([train_detection, test_detection], ignore_index=True)
detections.to_csv(OUTPUTS / 'metrics/detections.csv', index=False)
detection_summary = detections.groupby(['split','label']).agg(
    images=('detected','size'), detected=('detected','sum'), fallbacks=('fallback','sum'))
detection_summary['detection_rate'] = detection_summary.detected / detection_summary.images
detection_summary.to_csv(OUTPUTS / 'metrics/detection_summary.csv')
display(detection_summary)


## Train two classical classifiers

Split the training images into fit/validation partitions (80/20 by unique-image groups). Same-pixel duplicates cannot cross that boundary. Both classifiers receive the same embeddings and partitions. `StandardScaler` is fitted inside each training pipeline, never on validation/test images.

Start with `C=1.0`, seed 42, and explicit convergence checks. Select the deployment model by **validation macro-F1**, then refit both finalized models on the complete training split. No test-based model selection or hyperparameter tuning.


In [ ]:
fit_idx, val_idx = validation_partition(train_rows, seed=SEED)
print('Fit images:', len(fit_idx), '| Validation images:', len(val_idx))
models, selected_name, validation_scores = train_models(
    X_train, train_rows, fit_idx, val_idx, ARTIFACTS / 'models', seed=SEED)
display(validation_scores)
print('Selected using validation macro-F1:', selected_name)
print((ARTIFACTS / 'models/selection.json').read_text())
save_bundle(ARTIFACTS / 'models/emotion_pipeline.joblib', models[selected_name], extractor, selected_name)


## Milestone 3 Evaluation and reflection

Evaluate both finalized classifiers on exactly the same test images. All seven classes are included in macro metrics, including classes with no predicted examples. Matrices use true labels on rows and predictions on columns. Keep the validation-selected deployment model even if the other model has a better test score.


In [ ]:
test_scores, test_predictions = evaluate_models(models, X_test, test_rows, OUTPUTS / 'metrics')
display(test_scores)
from IPython.display import Image
for name in models:
    print(name)
    print((OUTPUTS / f'metrics/{name}_classification_report.txt').read_text())
    display(Image(filename=str(OUTPUTS / f'metrics/{name}_confusion_matrix.png')))
    print('Largest off-diagonal counts:')
    print((OUTPUTS / f'metrics/{name}_common_confusions.txt').read_text())
    fig = error_grid(test_predictions, name)
    if fig is not None:
        fig.savefig(OUTPUTS / f'figures/{name}_errors.png', dpi=150, bbox_inches='tight')
        plt.show()


### Reflection to complete after reviewing the outputs

For **each model**, write 2–3 sentences describing the most frequent confusions and the per-class recalls. Explain why accuracy and macro-F1 may tell different stories. Inspect errors before suggesting causes; possible ambiguity, pose, or crop problems are interpretations, not established causes.

For the **at-most-one-page comparative summary**, use the metrics table and 3–5 supported insights. Distinguish the validation-selected model from the higher test scorer if they differ. Discuss the limitations of generic ImageNet features, low resolution, detection fallbacks, and this dataset's labels. Do not invent personal reflections or team roles.


## Simple deployment and prediction proof

Reload the saved classifier bundle and reuse the identical feature configuration and weights. Demonstrate on two deterministically selected test images, never used for training or model selection. These are held-out test examples, not newly collected photographs. The helper checks that reloaded predictions match the evaluation pipeline.


In [ ]:
predictor = EmotionPredictor(ARTIFACTS / 'models/emotion_pipeline.joblib', extractor=extractor)
def predict_emotion(image_path, fer_style=True):
    return predictor.predict_emotion(image_path, fer_style=fer_style)

prediction_demo = demonstrate(predictor, test_rows, test_predictions, OUTPUTS / 'submission')
display(prediction_demo[['filename','predicted_label','true_label','detected','fallback','source']])
fig, axes = plt.subplots(1, len(prediction_demo), figsize=(10,4), squeeze=False)
import cv2
for ax, row in zip(axes.flat, prediction_demo.itertuples()):
    ax.imshow(cv2.imread(row.filename, cv2.IMREAD_GRAYSCALE), cmap='gray')
    ax.set_title(f'{Path(row.filename).name}\nPredicted: {row.predicted_label} | True: {row.true_label}', fontsize=9)
    ax.axis('off')
fig.tight_layout()
fig.savefig(OUTPUTS / 'figures/prediction_demo.png', dpi=160, bbox_inches='tight')
plt.show()

from src.reporting import write_summary
summary_path = write_summary(test_scores, selected_name, OUTPUTS, ARTIFACTS / 'audit')
print('Measured comparison draft:', summary_path)


## Submission evidence

Use `submission_checklist.md` and the provided Word submission form. Capture the split counts, both model configurations, both reports/matrices, and prediction table with filenames. Keep the executed notebook and `artifacts/environment.json` with the submission evidence.

**References**
- [FER2013 folder-format dataset](https://www.kaggle.com/datasets/msambare/fer2013)
- [MobileNetV2 configuration and preprocessing](https://keras.io/2/api/applications/mobilenet/)
- [Preventing preprocessing leakage](https://scikit-learn.org/stable/common_pitfalls.html)
- Supplied Week 7 laboratory and Week 7–8 lecture.
